# Study 1 Analysis
Code for analyzing data from the survey in Study 1

# Load packages

In [ ]:
import json
import os
import math
import pandas as pd
from scipy.stats import mannwhitneyu

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.options.mode.chained_assignment = None  # default='warn'

# Load Data

In [ ]:
survey_data_df = pd.read_csv(
    "./survey-data/final-survey-data_04-13-25.csv",
    keep_default_na=False,
)
print(json.dumps(list(survey_data_df.columns), indent=4))
survey_data_df.head()

In [ ]:
confidence_col = "Again, think about the AI tool you use most to caption products. When the tool says a photo is not good enough to caption or returns a similar error, how confident are you in knowing why the photo is not good enough?"


survey_data_df[survey_data_df[confidence_col] != ""][confidence_col].map(
    {
        "Extremely confident": 5,
        "Very confident": 4,
        "Somewhat confident": 3,
        "Slightly confident": 2,
        "Not at all confident": 1,
    }
).describe()

# Descriptive Stats

## Preferences for AI vs Human Assistance

In [ ]:
def get_descriptive_stats(df, col_dict, answer_dict, show_percent=False):
    """
    Generate a descriptive statistics DataFrame for scenario-based questions.

    Parameters:
        df (pd.DataFrame): The input DataFrame.
        col_dict (dict): Mapping of original column names to display names.
        answer_dict (dict): Mapping of answer text to values (order).
        show_percent (bool): Whether to include percentages in the output.

    Returns:
        pd.DataFrame: Descriptive statistics table.
    """
    result = pd.DataFrame(
        {x: "" for x in ["Answer"] + list(col_dict.values())}, index=[]
    )
    total_responses = len(df)
    for answer in answer_dict.keys():
        current_row = {"Answer": answer}
        for col in col_dict.keys():
            count = len(df[df[col] == answer])
            if show_percent:
                percent = count / total_responses if total_responses > 0 else 0
                current_row[col_dict[col]] = f"{count} ({percent:.1%})"
            else:
                current_row[col_dict[col]] = count
        result = pd.concat([result, pd.DataFrame([current_row])], ignore_index=True)
    return result


def plot_ai_human_side_by_side(
    left_df,
    right_df,
    left_title="Left",
    right_title="Right",
    answer_order=None,
    colors=None,
    wrap_width_left=20,
    wrap_width_right=20,
    min_label_threshold=5,
    figsize=(14, 5),
    font_family="sans-serif",
    font_size=11,
    center_label=None,
    share_xlim=True,
    xlim=None,
    legend_ncol=3,
):
    """
    Draw two diverging stacked horizontal bar charts side-by-side for comparison,
    each with its own y-axis labels and title, and a shared legend underneath.

    The shared legend is ordered so that it reads left-to-right across rows,
    e.g., with legend_ncol=3, the first row shows the first three categories
    in order, and the remainder appear on the second row.
    """
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=figsize)

    # Left subplot
    plot_ai_human_diverging_bars(
        left_df,
        answer_order=answer_order,
        colors=colors,
        title=left_title,
        wrap_width=wrap_width_left,
        min_label_threshold=min_label_threshold,
        figsize=figsize,
        font_family=font_family,
        font_size=font_size,
        center_label=center_label,
        ax=ax_l,
    )

    # Right subplot
    plot_ai_human_diverging_bars(
        right_df,
        answer_order=answer_order,
        colors=colors,
        title=right_title,
        wrap_width=wrap_width_right,
        min_label_threshold=min_label_threshold,
        figsize=figsize,
        font_family=font_family,
        font_size=font_size,
        center_label=center_label,
        ax=ax_r,
    )

    # Harmonize x-limits if requested
    if xlim is not None:
        ax_l.set_xlim(*xlim)
        ax_r.set_xlim(*xlim)
    elif share_xlim:
        l0, l1 = ax_l.get_xlim()
        r0, r1 = ax_r.get_xlim()
        max_abs = max(abs(l0), abs(l1), abs(r0), abs(r1))
        ax_l.set_xlim(-max_abs, max_abs)
        ax_r.set_xlim(-max_abs, max_abs)

    # Build legend handles/labels explicitly to control order (row-wise left-to-right)
    if colors is None:
        colors = [
            "#4B8BBE",
            "#89B4E0",
            "#E5E5E5",
            "#F5B97D",
            "#E07A5F",
        ]
    if answer_order is None:
        answer_order = [
            "Almost always just an AI based tool",
            "Most often just an AI based tool",
            "Both an AI based tool and human sighted assistance",
            "Most often just human sighted assistance",
            "Almost always just human sighted assistance",
        ]
    handles = [
        mpatches.Patch(color=color, label=label)
        for color, label in zip(colors, answer_order)
    ]
    labels = answer_order

    # Shared legend underneath; Matplotlib fills row-wise left-to-right across ncol
    ncols = legend_ncol if legend_ncol and legend_ncol > 0 else len(labels)
    new_handles = [
        handles[0],
        handles[3],
        handles[1],
        handles[4],
        handles[2],
        handles[3],
    ]
    new_labels = [
        labels[0],
        labels[3],
        labels[1],
        labels[4],
        labels[2],
    ]
    # Add a thin black border to each legend patch (use facecolor, not color, to avoid warning)
    bordered_handles = [
        mpatches.Patch(
            facecolor=patch.get_facecolor(),
            label=patch.get_label(),
            edgecolor="black",
            linewidth=0.1,
        )
        for patch in new_handles
    ]
    fig.legend(
        bordered_handles,
        new_labels,
        loc="lower center",
        ncol=ncols,
        frameon=False,
        bbox_to_anchor=(0.5, 0),
        fontsize=font_size - 1,
    )

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.15)
    return fig, (ax_l, ax_r)


def plot_ai_human_diverging_bars(
    dataframe,
    answer_order=None,
    colors=None,
    title="Distribution of AI vs Human Assistance Preferences",
    wrap_width=20,
    min_label_threshold=5,
    figsize=(8, 5),
    font_family="sans-serif",
    font_size=10,
    center_label=None,
    ax=None,
):
    """
    Render a diverging stacked horizontal bar chart for AI vs Human assistance preferences.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Either a wide DataFrame where the index are y-axis labels (scenarios) and columns
        are the five answer categories, or a long DataFrame containing a column named
        "Answer" with one row per category (in which case it will be pivoted via .set_index('Answer').T).
    answer_order : list[str] | None
        The ordered list of five category names from most-AI to most-human with the
        "Both" category in the middle. Defaults to the study's categories.
    colors : list[str] | None
        Five hex color codes matching answer_order. Defaults to the study's palette.
    title : str
        Plot title.
    wrap_width : int
        Number of characters to wrap y-axis labels to.
    min_label_threshold : int
        Minimum bar width (in count units) to draw a value label.
    figsize : tuple[int, int]
        Figure size for matplotlib.
    font_family : str
        Matplotlib font family to apply.
    font_size : int
        Base font size.
    center_label : str | None
        Optional override for the middle ("Both") label in the legend; defaults to answer_order[2].
    ax : matplotlib.axes.Axes | None
        Existing Axes to draw on; if None, a new Figure/Axes is created.

    Returns
    -------
    (fig, ax)
        Matplotlib Figure and Axes.
    """
    import matplotlib.pyplot as plt
    import textwrap
    from matplotlib.ticker import MaxNLocator, FuncFormatter

    # Defaults
    if answer_order is None:
        answer_order = [
            "Almost always just an AI based tool",
            "Most often just an AI based tool",
            "Both an AI based tool and human sighted assistance",
            "Most often just human sighted assistance",
            "Almost always just human sighted assistance",
        ]
    if colors is None:
        colors = [
            "#4B8BBE",  # blue: Almost always just an AI based tool
            "#89B4E0",  # light blue: Most often just an AI based tool
            "#E5E5E5",  # gray: Both an AI based tool and human sighted assistance
            "#F5B97D",  # light orange: Most often just human sighted assistance
            "#E07A5F",  # orange: Almost always just human sighted assistance
        ]
    if center_label is None:
        center_label = answer_order[2]

    # Prepare data (allow either long with "Answer" or wide with columns as categories)
    if "Answer" in dataframe.columns:
        plot_df = dataframe.set_index("Answer").T
    else:
        plot_df = dataframe.copy()

    # Validate and reorder columns to match answer_order
    missing = [a for a in answer_order if a not in plot_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    plot_df = plot_df[answer_order]

    # Reverse y order to match prior convention
    plot_df = plot_df.iloc[::-1]

    # Sort scenarios by AI-leaning totals (then flipped as in the original)
    ai_pref_totals = (
        plot_df[answer_order[0]].astype(int)
        + plot_df[answer_order[1]].astype(int)
        + plot_df[answer_order[2]].astype(int)
    )
    sorted_indices = ai_pref_totals.sort_values(ascending=False).index[::-1]
    plot_df_sorted = plot_df.loc[sorted_indices]

    # Y labels (wrapped)
    y_labels_sorted_wrapped = [
        "\n".join(textwrap.wrap(label, wrap_width))
        for label in plot_df_sorted.index.tolist()
    ]

    # Values
    v0 = plot_df_sorted[answer_order[0]].astype(int).values
    v1 = plot_df_sorted[answer_order[1]].astype(int).values
    v2 = plot_df_sorted[answer_order[2]].astype(int).values
    v3 = plot_df_sorted[answer_order[3]].astype(int).values
    v4 = plot_df_sorted[answer_order[4]].astype(int).values

    # Left offsets to center the middle (Both) bin at 0
    left0 = -v1 - v2 / 2 - v0
    left1 = -v2 / 2 - v1
    left2 = -v2 / 2
    left3 = v2 / 2
    left4 = v3 + v2 / 2

    # Create axes if needed
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    # Global style tweaks
    plt.rcParams["font.family"] = font_family
    plt.rcParams["font.size"] = font_size

    # Draw bars
    bars = []
    bars.append(
        ax.barh(
            y_labels_sorted_wrapped,
            v0,
            left=left0,
            color=colors[0],
            label=answer_order[0],
            zorder=3,
        )
    )
    bars.append(
        ax.barh(
            y_labels_sorted_wrapped,
            v1,
            left=left1,
            color=colors[1],
            label=answer_order[1],
            zorder=3,
        )
    )
    bars.append(
        ax.barh(
            y_labels_sorted_wrapped,
            v2,
            left=left2,
            color=colors[2],
            label=center_label,
            zorder=3,
        )
    )
    bars.append(
        ax.barh(
            y_labels_sorted_wrapped,
            v3,
            left=left3,
            color=colors[3],
            label=answer_order[3],
            zorder=3,
        )
    )
    bars.append(
        ax.barh(
            y_labels_sorted_wrapped,
            v4,
            left=left4,
            color=colors[4],
            label=answer_order[4],
            zorder=3,
        )
    )

    # Bar labels (only for sufficiently wide segments)
    for group in bars:
        for bar in group:
            width = bar.get_width()
            if abs(width) > min_label_threshold:
                x = bar.get_x() + width / 2
                y = bar.get_y() + bar.get_height() / 2
                ax.text(
                    x,
                    y,
                    f"{int(abs(width))}",
                    va="center",
                    ha="center",
                    color="black",
                    fontsize=font_size - 1,
                    zorder=4,
                    fontweight="semibold",
                )

    # Formatting
    ax.set_xlabel("")
    ax.set_ylabel("")
    if title:
        ax.set_title(
            textwrap.fill(title, width=40),
            fontsize=font_size + 1,
            fontweight="semibold",
        )

    ax.axvline(0, color="black", linewidth=0.8, zorder=1)
    ax.grid(axis="x", linestyle="-", zorder=2)

    # Symmetric x-limits and absolute-value tick labels
    # make them the max rounded to the nearest ceiling 5
    max_val = math.ceil(max(abs(ax.get_xlim()[0]), abs(ax.get_xlim()[1])) / 5) * 5
    ax.set_xlim(-max_val, max_val)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=10, integer=True))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(abs(x))}"))

    # remove spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.spines["left"].set_visible(False)

    fig.tight_layout()
    return fig, ax

In [ ]:
answer_dict = {
    "Almost always just human sighted assistance": 2,
    "Most often just human sighted assistance": 1,
    "Both an AI based tool and human sighted assistance": 0,
    "Most often just an AI based tool": -1,
    "Almost always just an AI based tool": -2,
}
scenario_cols = {
    "Which of the following are you most likely to use when searching for a specific product item at a physical store?": "searching for product physical store",
    "Which of the following are you most likely to use when browsing products at a physical store?": "browsing products at store",
    "Which of the following are you most likely to use when wanting to compare the details of two products side by side?": "comparing details of two products",
    "Which of the following are you most likely to use when identifying an unknown item in your home?": "id. unknown item in home",
    "Which of the following are you most likely to use when reading a label on a food item?": "reading food label",
    "Which of the following are you most likely to use when reading a label on medication?": "reading label on medication",
    "Which of the following are you most likely to use when identifying personal care products or toiletries?": "id. personal products / toiletries",
    "Which of the following are you most likely to use when checking expiration dates on products?": "checking expiration dates",
    "Which of the following are you most likely to use when checking allergen information on products?": "checking product allergen info",
}
display(
    get_descriptive_stats(
        survey_data_df[scenario_cols.keys()],
        scenario_cols,
        answer_dict,
        show_percent=True,
    )
)

concern_cols = {
    "Which of the following are you most likely to use when efficiency in identifying and understanding a product is your top concern?": "efficiency",
    "Which of the following are you most likely to use when data privacy in identifying and understanding a product is your top concern (e.g., the extent to which data you share is kept private)?": "data privacy",
    "Which of the following are you most likely to use when personal privacy in identifying and understanding a product is your top concern (e.g., for fear of embarrassment or stigma)?": "personal privacy",
    "Which of the following are you most likely to use when accuracy in identifying and understanding a product is your top concern?": "accuracy",
    "Which of the following are you most likely to use when financial cost in identifying and understanding a product is your top concern?": "financial cost",
    "Which of the following are you most likely to use when safety in identifying and understanding a product is your top concern?": "safety",
}
answer_dict = {
    "Almost always just human sighted assistance": 2,
    "Most often just human sighted assistance": 1,
    "Both an AI based tool and human sighted assistance": 0,
    "Most often just an AI based tool": -1,
    "Almost always just an AI based tool": -2,
}
display(
    get_descriptive_stats(
        survey_data_df[concern_cols.keys()],
        concern_cols,
        answer_dict,
        show_percent=True,
    )
)

In [ ]:
# Left/right can be long (has "Answer") or wide (columns == 5 categories)
fig, (ax_l, ax_r) = plot_ai_human_side_by_side(
    left_df=get_descriptive_stats(
        survey_data_df[scenario_cols.keys()],
        scenario_cols,
        answer_dict,
        show_percent=False,
    ),
    right_df=get_descriptive_stats(
        survey_data_df[concern_cols.keys()],
        concern_cols,
        answer_dict,
        show_percent=False,
    ),
    left_title=f"Scenario-Based Preference for AI vs Human Assistance (N = {len(survey_data_df[['Timestamp'] + list(scenario_cols.keys())]['Timestamp'].unique())})",
    right_title=f"Concern-Based Preference for AI vs Human Assistance (N = {len(survey_data_df[['Timestamp'] + list(concern_cols.keys())]['Timestamp'].unique())})",
    figsize=(12, 6),
    font_size=11,
    colors=[
        "#ca0020",  # blue: Almost always just an AI based tool
        "#f4a582",  # light blue: Most often just an AI based tool
        "#f7f7f7",  # gray: Both an AI based tool and human sighted assistance
        "#92c5de",  # light orange: Most often just human sighted assistance
        "#0571b0",  # orange: Almost always just human sighted assistance
    ],
    wrap_width_left=21,
    wrap_width_right=10,
)
# save as pdf
os.makedirs("./plots", exist_ok=True)
fig.savefig(
    "./plots/scenario-vs-concern-preference.pdf", bbox_inches="tight", pad_inches=0
)

## Confidence

In [ ]:
confidence_col = "Again, think about the AI tool you use most to caption products. When the tool says a photo is not good enough to caption or returns a similar error, how confident are you in knowing why the photo is not good enough?"


survey_data_df[survey_data_df[confidence_col] != ""][confidence_col].map(
    {
        "Extremely confident": 5,
        "Very confident": 4,
        "Somewhat confident": 3,
        "Slightly confident": 2,
        "Not at all confident": 1,
    }
).describe()

# Statistical analysis 

In [ ]:
image_quality_factor_cols = [
    "Keeping the tool you use most in mind, how much does lighting in your environment affect the quality of AI generated captions for products?",
    "How much does taking clear, non-blurry photos affect the quality of AI generated captions for products?",
    "How much does having the object in full view of your camera, rather than a partial view of the object, affect the quality of AI generated captions for products?",
    "How much does rotating your camera or the object affect the quality of AI generated captions for products?",
    "How much does your hand placement or how you are holding the object affect the quality of AI generated captions for products?",
    "How much does moving your camera closer or further from an object affect the quality of AI generated captions for products?",
    "How much does a product being uncommon or unique affect the quality of AI generated captions?",
]

image_assessment_factor_cols = [
    "How well does the AI tool help you assess lighting conditions when taking a photo?",
    "How well does the AI tool help you assess whether the photo you took is clear and non-blurry?",
    "How well does the AI tool help you assess whether an object is in full view of your camera when taking photos?",
    "How well does the AI tool help you assess object orientation when taking photos?",
    "How well does the AI tool help you understand your hand positioning or placement relative to the object you want to photograph?",
    "How well does the AI tool help you assess the distance between your camera and the object of interest when taking photos?",
    "How well does the AI tool help you assess whether a product you are taking a photo of is uncommon or unique?",
]


mapping_dict = {
    "To a great extent": 5,
    "Somewhat": 4,
    "Very little": 3,
    "Not at all": 2,
    "I am not sure": 1,
}

In [ ]:
condition_col = "Which of the following AI based tools do you use the most to identify and understand products?"
survey_data_df[condition_col].value_counts()

In [ ]:
len(survey_data_df)

In [ ]:
# calculate mean and statistical differences for how much image quality affects response
for question in image_quality_factor_cols:
    # create dataframes for each condition
    be_my_ai_products_df = survey_data_df[
        survey_data_df[condition_col]
        == "Be My AI (part of Be My Eyes that does not involve human visual intepreters)"
    ]
    seeing_ai_products_df = survey_data_df[
        survey_data_df[condition_col] == "Microsoft Seeing AI"
    ]

    # convert to number
    seeing_ai_products_df[question] = seeing_ai_products_df[question].map(mapping_dict)
    be_my_ai_products_df[question] = be_my_ai_products_df[question].map(mapping_dict)

    # remove rows where any of the image quality factor columns are empty
    seeing_ai_products_df = seeing_ai_products_df[
        seeing_ai_products_df[image_quality_factor_cols].notna().all(axis=1)
    ]
    be_my_ai_products_df = be_my_ai_products_df[
        be_my_ai_products_df[image_quality_factor_cols].notna().all(axis=1)
    ]

    print(
        f"Be My AI: n = {len(be_my_ai_products_df)} | Seeing AI: n = {len(seeing_ai_products_df)}"
    )

    U, p = mannwhitneyu(
        seeing_ai_products_df[question],
        be_my_ai_products_df[question],
        nan_policy="omit",
    )
    print(f"{question}")
    print(
        f"Mean Seeing AI: {seeing_ai_products_df[question].mean():.2f} | Mean Be My AI: {be_my_ai_products_df[question].mean():.2f}",
        end="",
    )
    if p <= 0.05:
        print(" -- STATISTICALLY SIGNIFICANT")
    else:
        print()
    print(f"U: {U}, p: {p:.4f}", end="")
    if p < 0.001:
        print("***", end="")
    elif p < 0.01:
        print("**", end="")
    elif p < 0.05:
        print("*", end="")
    elif p < 0.1:
        print(".", end="")
    print("\n")

### Differences in Percieved Impact between Be My AI and Seeing AI 

In [ ]:
print("Differences in Percieved Impact between Be My AI and Seeing AI")
print("-" * 80)

# create dataframes for each condition
be_my_ai_products_df = survey_data_df[
    survey_data_df[condition_col]
    == "Be My AI (part of Be My Eyes that does not involve human visual intepreters)"
]
seeing_ai_products_df = survey_data_df[
    survey_data_df[condition_col] == "Microsoft Seeing AI"
]

for question in image_quality_factor_cols:
    # convert to number
    seeing_ai_products_df[question] = seeing_ai_products_df[question].map(mapping_dict)
    be_my_ai_products_df[question] = be_my_ai_products_df[question].map(mapping_dict)

# remove rows where any of the image quality factor columns are empty
seeing_ai_products_df = seeing_ai_products_df[
    seeing_ai_products_df[image_quality_factor_cols].notna().all(axis=1)
]
be_my_ai_products_df = be_my_ai_products_df[
    be_my_ai_products_df[image_quality_factor_cols].notna().all(axis=1)
]

# print sample size
print(
    f"Be My AI: n = {len(be_my_ai_products_df)} | Seeing AI: n = {len(seeing_ai_products_df)}"
)

# calculate mean and statistical differences for how much the tool supports diagnosing issues
for question in image_quality_factor_cols:
    # filter out 1 responses ("I am not sure")
    seeing_ai_products_df = seeing_ai_products_df[seeing_ai_products_df[question] != 1]
    be_my_ai_products_df = be_my_ai_products_df[be_my_ai_products_df[question] != 1]

    U, p = mannwhitneyu(
        seeing_ai_products_df[question],
        be_my_ai_products_df[question],
        nan_policy="omit",
    )
    print(f"{question}")
    print(
        f"Mean Seeing AI: {seeing_ai_products_df[question].mean():.2f} | Mean Be My AI: {be_my_ai_products_df[question].mean():.2f}",
        end="",
    )
    if p <= 0.05:
        print(" -- STATISTICALLY SIGNIFICANT")
    else:
        print()

    print(f"p: {p:.4f}, U: {U}", end="")
    if p < 0.001:
        print("***", end="")
    elif p < 0.01:
        print("**", end="")
    elif p < 0.05:
        print("*", end="")
    elif p < 0.1:
        print(".", end="")

    # for seeing_ai_products_df, print the number of responses for each category (1-5)
    categories = [1, 2, 3, 4, 5]
    counts_seeing_ai = (
        seeing_ai_products_df[question].value_counts().reindex(categories, fill_value=0)
    )
    counts_be_my_ai = (
        be_my_ai_products_df[question].value_counts().reindex(categories, fill_value=0)
    )
    counts_df = pd.DataFrame(
        {
            "Rating": categories,
            "Seeing AI": counts_seeing_ai.values,
            "Be My AI": counts_be_my_ai.values,
        }
    )
    counts_df.loc["Total"] = counts_df.sum()
    display(counts_df)

    print("\n")

### Differences in Assessment Ability between Be My AI and Seeing AI

In [ ]:
print("Differences in Assessment Ability between Be My AI and Seeing AI")
print("-" * 80)

# create dataframes for each condition
be_my_ai_products_df = survey_data_df[
    survey_data_df[condition_col]
    == "Be My AI (part of Be My Eyes that does not involve human visual intepreters)"
]
seeing_ai_products_df = survey_data_df[
    survey_data_df[condition_col] == "Microsoft Seeing AI"
]

for question in image_assessment_factor_cols:
    # convert to number
    seeing_ai_products_df[question] = seeing_ai_products_df[question].map(mapping_dict)
    be_my_ai_products_df[question] = be_my_ai_products_df[question].map(mapping_dict)

# remove rows where any of the image quality factor columns are empty
seeing_ai_products_df = seeing_ai_products_df[
    seeing_ai_products_df[image_assessment_factor_cols].notna().all(axis=1)
]
be_my_ai_products_df = be_my_ai_products_df[
    be_my_ai_products_df[image_assessment_factor_cols].notna().all(axis=1)
]

# print sample size
print(
    f"Be My AI: n = {len(be_my_ai_products_df)} | Seeing AI: n = {len(seeing_ai_products_df)}"
)

# calculate mean and statistical differences for how much the tool supports diagnosing issues
for question in image_assessment_factor_cols:
    # filter out 1 responses ("I am not sure")
    seeing_ai_products_df = seeing_ai_products_df[seeing_ai_products_df[question] != 1]
    be_my_ai_products_df = be_my_ai_products_df[be_my_ai_products_df[question] != 1]

    U, p = mannwhitneyu(
        seeing_ai_products_df[question],
        be_my_ai_products_df[question],
        nan_policy="omit",
    )
    print(f"{question}")
    print(
        f"Mean Seeing AI: {seeing_ai_products_df[question].mean():.2f} | Mean Be My AI: {be_my_ai_products_df[question].mean():.2f}",
        end="",
    )
    if p <= 0.05:
        print(" -- STATISTICALLY SIGNIFICANT")
    else:
        print()

    print(f"p: {p:.4f}, U: {U}", end="")
    if p < 0.001:
        print("***", end="")
    elif p < 0.01:
        print("**", end="")
    elif p < 0.05:
        print("*", end="")
    elif p < 0.1:
        print(".", end="")
    print("\n")

    # for seeing_ai_products_df, print the number of responses for each category (1-5)
    categories = [1, 2, 3, 4, 5]
    counts_seeing_ai = (
        seeing_ai_products_df[question].value_counts().reindex(categories, fill_value=0)
    )
    counts_be_my_ai = (
        be_my_ai_products_df[question].value_counts().reindex(categories, fill_value=0)
    )
    counts_df = pd.DataFrame(
        {
            "Rating": categories,
            "Seeing AI": counts_seeing_ai.values,
            "Be My AI": counts_be_my_ai.values,
        }
    )
    counts_df.loc["Total"] = counts_df.sum()
    display(counts_df)

    print("\n")